# Defining the ERI tensor from checkpoint directly
In the integral evaluation section, at this point we are able to generate the ERI tensor, but was done as: 

```python 
fullbasistr = """
    H     S
        1.0    1
    H     P
        2.0    1
    H     D
        3.0    1
"""

r1 = np.array([0.0, 0.0, 0.0])
Te1_primitives = primitives_from_basis_string(r1, "Te", fullbasistr)

gto_list = [*Te1_primitives, ]
Eri_test: NDArray[np.float64] = naive_wrapper_interest(gto_list, None, symm=4)
```

And while this works, we need a helper function to extract exacly the data from the checkpoint basis definition.

In [1]:
from typing import Literal

import h5py

import numpy as np
from numpy.typing import NDArray

from py_mods.src.integrals.GTO import create_GTO

from py_mods.src.integrals.UncontractedBasisSet import (
    create_UncontractedBasisSet,
    ERIs_Uncontracted,
)

In [2]:
checkpoint_files = [
    "files/H_checkpoint.h5",
    "files/He_checkpoint.h5",
    "files/Ne_checkpoint.h5",
]

In [3]:
def extract_basis_data(h5filename: str, component: Literal["Large", "Small"] = "Large"):

    if component not in ["Large", "Small"]:
        raise ValueError("Component must be either 'Large' or 'Small'")

    basis_number = "1" if component == "Large" else "2"

    with h5py.File(h5filename, "r") as f:
        R_array = np.asarray(f[f"input/aobasis/{basis_number}/center"][()])
        R_array = R_array.reshape(-1, 3)
        exps_array = np.asarray(f[f"input/aobasis/{basis_number}/exponents"][()])
        l_array = np.asarray(f[f"input/aobasis/{basis_number}/orbmom"][()])

    return R_array, exps_array, l_array

In [4]:
print(extract_basis_data(checkpoint_files[0], 'Large'))
print(extract_basis_data(checkpoint_files[0], 'Small'))

(array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]]), array([82.9687389 , 12.4571508 ,  2.83382422,  0.80016414,  0.25862944,
        0.08997668,  0.5024489 ]), array([1, 1, 1, 1, 1, 1, 2]))
(array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]]), array([ 0.5024489 , 82.9687389 , 12.4571508 ,  2.83382422,  0.80016414,
        0.25862944,  0.08997668,  0.5024489 ]), array([1, 2, 2, 2, 2, 2, 2, 3]))


In [5]:
def build_uncontracted_basis_from_checkpoint(h5filename):
    Ldata = extract_basis_data(h5filename, "Large")
    Sdata = extract_basis_data(h5filename, "Small")

    LC_list = []

    for R, alpha, l in zip(*Ldata):
        gto_instance = create_GTO(R, alpha, l-1)
        LC_list.append(gto_instance)
    
    SC_list = []
    for R, alpha, l in zip(*Sdata):
        gto_instance = create_GTO(R, alpha, l-1)
        SC_list.append(gto_instance)
    
    total_basis = LC_list + SC_list

    Unc_bas_set = create_UncontractedBasisSet(total_basis)

    return Unc_bas_set

In [6]:
h_basis = build_uncontracted_basis_from_checkpoint(checkpoint_files[0])
sum(h_basis.l_dims)

np.int16(34)

In [7]:
eri_tensor = ERIs_Uncontracted(h_basis)

In [8]:
def eri_alpha_beta(i, j, k, l, eri_tensor, size=None):
    """Returns the (i,j,k,l) element of the ERI tensor, using modulo to wrap indices since alpha and beta ERIS are equal."""
    if size is not None:
        pass

    else:
        size = eri_tensor.shape[0]

    return eri_tensor[i % size, j % size, k % size, l % size]

In [9]:
eri_alpha_beta(34,34,34,34, eri_tensor) == eri_tensor[0,0,0,0]

np.True_